In [345]:
import pandas as pd
import numpy as np
from pathlib import Path

---------------------------------------------------------------------------------------------------------------------------------------------
## Загрузка данных
---------------------------------------------------------------------------------------------------------------------------------------------

In [347]:
DATA_PATH = Path("../data/raw")

tables = {}
for file in DATA_PATH.glob('*.csv'):
    tables[file.stem] = pd.read_csv(file)

In [349]:
sellers_df = tables['olist_sellers_dataset']
product_category_df = tables['product_category_name_translation']
orders_df = tables['olist_orders_dataset']
order_items_df = tables['olist_order_items_dataset']
customers_df = tables['olist_customers_dataset']
geolocation_df = tables['olist_geolocation_dataset']
order_payments_df = tables['olist_order_payments_dataset']
order_reviews_df = tables['olist_order_reviews_dataset']
products_df = tables['olist_products_dataset']

In [385]:
dfs = {
    'sellers_df': sellers_df,
    'product_category_df': product_category_df,
    'orders_df': orders_df,
    'order_items_df': order_items_df,
    'customers_df': customers_df, 
    'geolocation_df': geolocation_df,
    'order_payments_df': order_payments_df,
    'order_reviews_df': order_reviews_df,
    'products_df': products_df
}

---------------------------------------------------------------------------------------------------------------------------------------------
## Анализ пропущенных значений
Пропуски были проанализированы на предыдущем этапе. По результатам анализа удаление или заполнение не выполнялось, 
так как они обусловлены бизнес-процессами.
### olist_orders_dataset
Большая часть пропусков даты подтверждения заказа (order_approved_at) относится к отменённым заказам, что соответствует бизнес-логике. Однако небольшое количество доставленных заказов также не содержит дату подтверждения оплаты, что может свидетельствовать о неполноте данных.
Пропуски даты передачи заказа перевозчику в основном относятся к заказам, которые были отменены или не дошли до этапа отправки. Такие пропуски являются ожидаемыми и отражают состояние заказа. Отсутствие даты доставки характерно для заказов, которые ещё находятся в пути, были отменены или недоступны для выполнения. Данные пропуски объясняются жизненным циклом заказа и не являются ошибкой.

Обнаружено 14 доставленных заказов без даты подтверждения оплаты. Поскольку доля таких записей крайне мала, значения не корректировались, но были отмечены как потенциальная аномалия данных.
### olist_order_reviews_dataset
Поле review_comment_title содержит большое количество пропусков, что объясняется необязательностью заполнения заголовка при оставлении отзыва. Пропуски не рассматриваются как ошибка данных и не требуют обработки. Большинство пропусков текстовых комментариев сопровождаются высокими оценками (4–5 баллов). Вероятно, пользователи чаще оставляют только числовую оценку без текстового комментария. Такие пропуски не являются ошибкой данных.
### olist_products_dataset
Пропуски в характеристиках товаров наблюдаются у небольшой части записей и, вероятно, связаны с отсутствием информации, предоставленной продавцом. На этапе очистки данные не заполнялись, поскольку достоверно восстановить значения невозможно.

---------------------------------------------------------------------------------------------------------------------------------------------
## Анализ и очистка дубликатов
Полные дубликаты в таблицах отсутствуют. Проверка уникальных идентификаторов показала, что структура данных соответствует ожидаемой модели: первичные ключи уникальны, а повторения внешних ключей обусловлены связями между таблицами (например, одному заказу может соответствовать несколько товаров или платежей).
Дубликаты в geolocation связаны с тем, что координаты одного ZIP-кода повторяются, поэтому оставим только уникальные комбинации ZIP + координаты.

In [353]:
geolocation_df = geolocation_df.drop_duplicates(
    subset=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
)

---------------------------------------------------------------------------------------------------------------------------------------------
## Преобразование типов
---------------------------------------------------------------------------------------------------------------------------------------------

In [355]:
date_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

date_cols_order_items = ['shipping_limit_date']

date_cols_reviews = [
    'review_creation_date',
    'review_answer_timestamp'
]

orders_df[date_cols_orders] = orders_df[date_cols_orders].apply(pd.to_datetime)
order_items_df[date_cols_order_items] = order_items_df[date_cols_order_items].apply(pd.to_datetime)
order_reviews_df[date_cols_reviews] = order_reviews_df[date_cols_reviews].apply(pd.to_datetime)

---------------------------------------------------------------------------------------------------------------------------------------------
## Добавление новой информации в таблицы
---------------------------------------------------------------------------------------------------------------------------------------------

### Информация о сроках доставки, сроков обработки, задержках, полной сумме заказа

In [357]:
orders_df['approval_time_hours'] = (
    (orders_df['order_approved_at'] - orders_df['order_purchase_timestamp']).dt.total_seconds() / 3600
).round(2)

orders_df['delivered_time_days'] = (orders_df['order_delivered_customer_date'] - orders_df['order_purchase_timestamp']).dt.days

orders_df['delivery_delay_days'] = (orders_df['order_delivered_customer_date'] - orders_df['order_estimated_delivery_date']).dt.days

orders_df['is_delay'] = (orders_df['delivery_delay_days'] > 0).astype('int')

payments_agg = order_payments_df.groupby('order_id', as_index=False)['payment_value'].sum()
payments_agg = payments_agg.merge(orders_df, how='right', on='order_id')
orders_df['order cost'] = payments_agg['payment_value']

---------------------------------------------------------------------------------------------------------------------------------------------
## Объединение таблиц заказов, платежей и покупателей для удобства дальнейшего анализа
---------------------------------------------------------------------------------------------------------------------------------------------

In [359]:
orders_full_df = orders_df.merge(order_payments_df, how='left', on='order_id')
orders_full_df = orders_full_df.merge(customers_df, how='left', on='customer_id')

dfs['orders_full_df'] = orders_full_df

---------------------------------------------------------------------------------------------------------------------------------------------
## Изменение порядка столбцов таблицы
---------------------------------------------------------------------------------------------------------------------------------------------

In [361]:
cols = [
    # IDs
    'order_id', 'customer_id', 'customer_unique_id',
    # даты
    'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date',
    # метрики доставки
    'approval_time_hours', 'delivered_time_days', 'is_delay',
    # финансовые метрики
    'payment_type', 'payment_installments', 'payment_value',
    # географические данные покупателя
    'customer_zip_code_prefix', 'customer_city', 'customer_state',
    # статус заказа
    'order_status'
]


orders_full_df = orders_full_df[cols]

---------------------------------------------------------------------------------------------------------------------------------------------
## Финальная проверка данных после очистки
---------------------------------------------------------------------------------------------------------------------------------------------

In [375]:
def final_quality_check(df):
    print('Пропуски:')
    display(df.isna().sum()[df.isna().sum() > 0])
    print('Дубликаты:')
    display(df.duplicated().sum())
    print('Размеры:', df.shape)
    print()

for name, df in dfs.items():
    final_quality_check(df)

Пропуски:


Series([], dtype: int64)

Дубликаты:


0

Размеры: (3095, 4)

Пропуски:


...

Размеры: (32951, 9)

Пропуски:


order_approved_at                 175
order_delivered_carrier_date     1888
order_delivered_customer_date    3132
approval_time_hours               175
delivered_time_days              3132
delivery_delay_days              3132
order cost                          1
payment_sequential                  1
payment_type                        1
payment_installments                1
payment_value                       1
dtype: int64

Дубликаты:


0

Размеры: (103887, 21)



---------------------------------------------------------------------------------------------------------------------------------------------
## Сохранение очищенных данных
---------------------------------------------------------------------------------------------------------------------------------------------

In [381]:
OUTPUT_PATH = Path("../data/cleaned")
dfs = {name.replace('_df', '_clean') if '_df' in name 
       else name if name.endswith('_clean') 
       else f"{name}_clean": df 
       for name, df in dfs.items()}

for name, df in dfs.items():
    df.to_csv(OUTPUT_PATH / f'{name}.csv', index=False)

## Вывод

На этапе подготовки данных были выполнены:

- проверка структуры таблиц;
- анализ пропусков;
- анализ дубликатов;
- преобразование типов данных;
- объединение необходимых таблиц для последующего анализа;
- изменение порядка столбцов;
- финальная проверка качества данных;
- сохранение обработанных датасетов.

Полученные данные готовы для проведения исследовательского анализа (EDA), расчёта продуктовых метрик и поиска бизнес-инсайтов.